<a href="https://colab.research.google.com/github/crialejo24/DOWNSCALING/blob/main/ESRGAN_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load libraries and Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q huggingface_hub

from huggingface_hub import HfApi
import os
import subprocess


Mounted at /content/drive


# Section 1: New training

## Load official repository

In [ ]:
!git clone https://github.com/xinntao/BasicSR

%cd BasicSR

!pip install -r requirements.txt
!python setup.py develop

Cloning into 'BasicSR'...
remote: Enumerating objects: 5924, done.
remote: Total 5924 (delta 0), reused 0 (delta 0), pack-reused 5924 (from 1)
Receiving objects: 100% (5924/5924), 4.14 MiB | 20.87 MiB/s, done.
Resolving deltas: 100% (3759/3759), done.
/content/BasicSR
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 143.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 28.2 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/content/BasicSR/setup.py", line 149, in <module>
    version=get_version(),
            ~~~~~~~~~~~^^
  File "/content/BasicSR/setup.py", line 79, in get_version
    return locals()['__version__']
           ~~~~~~~~^^^^^^^^^^^^^^^
KeyError: '__version__'


## Select training type

You can find the two fine-tuning training configurations shown in the figure in the "Downscaling" GitHub repository by author crialejo24.

![ESRGAN](https://imgur.com/MSUY8bZ.jpg)

In [ ]:
training='training1'
#training='training2'

## Replace modified files

In [ ]:
import os
import urllib.request

def select_training_type(training):

    # Verificar selección
    if training == "training1":
        carpeta = "modificables_esrgan_train_1"
    elif training == "training2":
        carpeta = "modificables_esrgan_train_2"
    else:
        raise ValueError("La opción debe ser 'training1' o 'training2'.")

    # Archivos que se van a descargar
    archivos = {
        "img_util.py": f"https://raw.githubusercontent.com/crialejo24/DOWNSCALING/main/{carpeta}/img_util.py",
        "rrdbnet_arch.py": f"https://raw.githubusercontent.com/crialejo24/DOWNSCALING/main/{carpeta}/rrdbnet_arch.py",
        "sr_model.py": f"https://raw.githubusercontent.com/crialejo24/DOWNSCALING/main/{carpeta}/sr_model.py"
    }

    # Destinos dentro de BasicSR

    destinos = {
        "img_util.py": "/content/BasicSR/basicsr/utils/img_util.py",
        "rrdbnet_arch.py": "/content/BasicSR/basicsr/archs/rrdbnet_arch.py",
        "sr_model.py": "/content/BasicSR/basicsr/models/sr_model.py"
    }


    # Descargar archivos
    for archivo, url in archivos.items():
        destino = destinos[archivo]

        print(f"Descargando {archivo} desde {carpeta}...")

        urllib.request.urlretrieve(url, destino)

        print(f"✓ {archivo} actualizado")

    print(f"\nConfiguración seleccionada: {training}")

In [ ]:
select_training_type(training)

Descargando img_util.py desde modificables_esrgan_train_1...
✓ img_util.py actualizado
Descargando rrdbnet_arch.py desde modificables_esrgan_train_1...
✓ rrdbnet_arch.py actualizado
Descargando sr_model.py desde modificables_esrgan_train_1...
✓ sr_model.py actualizado

Configuración seleccionada: training1


### If you need to modify the trainable layers, load either of the two configurations and edit them in the following file: /content/BasicSR/basicsr/models/sr_model.py

## Upload dataset images

In [ ]:
!mkdir -p datasets/train/HR
!mkdir -p datasets/train/LR
!mkdir -p datasets/test/HR
!mkdir -p datasets/test/LR

api = HfApi()

for hf, dst in [
    ("trainset/HR_192_mod", "/content/BasicSR/datasets/train/HR"),
    ("trainset/LR_64_modx3", "/content/BasicSR/datasets/train/LR"),
    ("testset/HR_192_mod", "/content/BasicSR/datasets/test/HR"),
    ("testset/LR_64_modx3", "/content/BasicSR/datasets/test/LR")
]:
    os.makedirs(dst, exist_ok=True)

    for f in api.list_repo_tree(
        "Crialejo924/DOWNSCALING",
        path_in_repo=hf,
        repo_type="dataset",
        recursive=True
    ):
        if hasattr(f, "path") and f.path.lower().endswith((".tif", ".tiff")):

            subprocess.run([
                "wget",
                "-q",
                "--show-progress",
                f"https://huggingface.co/datasets/Crialejo924/DOWNSCALING/resolve/main/{f.path}",
                "-P",
                dst
            ])

KeyboardInterrupt: 

## Load pre-trained models

In [ ]:
!python scripts/download_pretrained_models.py ESRGAN

## Training Configuration

It is important to assign a path external to Colab to the "path" item in the .yaml file in order to save training progress and avoid losing it if the Colab session closes or restarts. By default, the user's Google Drive will be used.

You must also specify the path to the pre-trained model in the "pretrain_network_g" item.

In [ ]:
progress_folder = f'/content/drive/MyDrive/Repository_ESRGAN_{training}'

os.makedirs(progress_folder, exist_ok=True)

In [ ]:
import yaml

archivo = "/content/BasicSR/options/train/ESRGAN/train_RRDBNet_PSNR_x4.yml"

config = {
    "name": "051_RRDBNet_PSNR_x4_f64b23_DIV2K_1000k_B16G1_wandb",
    "model_type": "SRModel",
    "scale": 3,
    "num_gpu": 1,
    "manual_seed": 0,

    "datasets": {
        "train": {
            "name": "DIV2K",
            "type": "PairedImageDataset",
            "dataroot_gt": "datasets/train/HR",
            "dataroot_lq": "datasets/train/LR",
            "filename_tmpl": "{}",
            "io_backend": {"type": "disk"},
            "gt_size": 192,
            "use_hflip": True,
            "use_rot": True,
            "num_worker_per_gpu": 4,
            "batch_size_per_gpu": 8,
            "dataset_enlarge_ratio": 100,
            "prefetch_mode": None
        },
        "val": {
            "name": "Set5",
            "type": "PairedImageDataset",
            "dataroot_gt": "datasets/test/HR",
            "dataroot_lq": "datasets/test/LR",
            "io_backend": {"type": "disk"}
        }
    },

    "network_g": {
        "type": "RRDBNet",
        "num_in_ch": 3,
        "num_out_ch": 3,
        "num_feat": 64,
        "num_block": 23,
        "num_grow_ch": 32,
        "scale": 3
    },

    "path": {
        "experiments_root": f"{progress_folder}/experiments/ESRGAN_{training}",
        "models": f"{progress_folder}/experiments/ESRGAN_{training}/models",
        "training_states": f"{progress_folder}/experiments/ESRGAN_{training}/training_states",
        "log": f"{progress_folder}/experiments/ESRGAN_{training}",
        "visualization": f"{progress_folder}/experiments/ESRGAN_{training}/visualization",

        "pretrain_network_g": "/content/BasicSR/experiments/pretrained_models/ESRGAN/ESRGAN_PSNR_SRx4_DF2K_official-150ff491.pth",
        "strict_load_g": True,
        "resume_state": None
    },

    "train": {
        "ema_decay": 0.999,

        "optim_g": {
            "type": "Adam",
            "lr": 2e-4,
            "weight_decay": 0,
            "betas": [0.9, 0.99]
        },

        "scheduler": {
            "type": "CosineAnnealingRestartLR",
            "periods": [250000, 250000, 250000, 250000],
            "restart_weights": [1, 1, 1, 1],
            "eta_min": 1e-7
        },

        "total_iter": 1000000,
        "warmup_iter": -1,

        "pixel_opt": {
            "type": "L1Loss",
            "loss_weight": 1.0,
            "reduction": "mean"
        }
    },

    "val": {
        "val_freq": 5000,
        "save_img": True,

        "metrics": {
            "psnr": {
                "type": "calculate_psnr",
                "crop_border": 4,
                "test_y_channel": False
            }
        }
    },

    "logger": {
        "print_freq": 100,
        "save_checkpoint_freq": 5000,
        "use_tb_logger": True,

        "wandb": {
            "project": None,
            "resume_id": None
        }
    },

    "dist_params": {
        "backend": "nccl",
        "port": 29500
    }
}

with open(archivo, "w") as f:
    yaml.dump(
        config,
        f,
        sort_keys=False,
        default_flow_style=False
    )

print("Archivo actualizado correctamente.")

## Save repository with dataset and training configuration to Google Drive

In [ ]:
!cp -r /content/BasicSR* "{progress_folder}"

## Execution of the training

In [ ]:
!python /content/BasicSR/basicsr/train.py -opt /content/BasicSR/options/train/ESRGAN/train_RRDBNet_PSNR_x4.yml

#Section 2: Resume training from a repository saved in Drive.

The following section allows you to resume a training session that was started previously and saved to Drive. Verify that the Google Drive folder name is correct.

## Load repository

In [ ]:
folder=f'/content/drive/MyDrive/Repository_ESRGAN_{training}'

#Training repository
!cp -r "{folder}" /content/BasicSR
%cd BasicSR

!pip install -r requirements.txt
!python setup.py develop

## Resume training

In [ ]:
!python /content/BasicSR/basicsr/train.py -opt /content/BasicSR/options/train/ESRGAN/train_RRDBNet_PSNR_x4.yml